# Extension 1 Demo

End-to-end walkthrough of the **Extension 1** pipeline on three synthetic sales scenarios.

| Scenario | Description |
|---|---|
| **Growth** | Strong upward trend — product in launch/growth phase |
| **Decline** | Strong downward trend — product losing market share |
| **Volatile** | No clear trend, high noise — unpredictable market |

**Covariates** (same for all scenarios): `marketing_spend`, `promo_discount`, `competitor_sales`

**Pipeline outputs per scenario**
- P10 / P50 / P90 quantile forecast with covariate overlay
- Covariate attribution via Attention Rollout
- Temporal saliency — which history steps each covariate attended to most
- Natural-language verbalization with per-sentence NLI consistency scores

In [ ]:
!git clone https://github.com/Voyagers-time-series-forecasting/explainable-chronos.git
%cd explainable-chronos
!git checkout dev-emanuele2
!pip install --upgrade pip setuptools wheel
!pip install -e .

In [ ]:
import sys, pathlib, warnings, textwrap
warnings.filterwarnings("ignore")

repo_root = pathlib.Path().resolve()
if not (repo_root / "extension_1").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

from extension_1.pipeline import VerbalizationPipeline
from extension_1.config import PipelineConfig
from extension_1.evaluation.scoring import NLIConsistencyScorer
from extension_1.verbalization.template import TemplateVerbalizer
from extension_1.attribution.attention import AttentionAttributor
from extension_1.attribution.types import CovariateSet
from shared.forecast_provider import ChronosForecastProvider

print("Imports OK")

In [ ]:
# ── Helper functions (data splitting + visualisation) ────────────────────

COV_NAMES  = ["marketing_spend", "promo_discount", "competitor_sales"]
_CMAP      = plt.colormaps.get_cmap("tab10")
COV_COLORS = {n: _CMAP(i) for i, n in enumerate(COV_NAMES)}
_NLI_GREEN  = "#2ecc71"
_NLI_YELLOW = "#f39c12"
_NLI_RED    = "#e74c3c"


def make_split(target, mkt, promo, comp, N, HORIZON):
    def _s(arr): return arr[:N], arr[N:N+HORIZON]
    h, fh = _s(target)
    m, fm = _s(mkt)
    p, fp = _s(promo)
    c, fc = _s(comp)
    past   = {"marketing_spend": m,  "promo_discount": p,  "competitor_sales": c}
    future = {"marketing_spend": fm, "promo_discount": fp, "competitor_sales": fc}
    return h, fh, past, future


def _nli_color(s):
    return _NLI_GREEN if s >= 0.70 else (_NLI_YELLOW if s >= 0.50 else _NLI_RED)


def plot_scenario(result, history, future, cov_set, scenario_name, HORIZON, tail=80):
    import matplotlib.patches as mpatches
    fig = plt.figure(figsize=(17, 15), constrained_layout=True)
    fig.suptitle(f"Scenario: {scenario_name}  —  daily unit sales",
                 fontsize=13, fontweight="bold")
    gs = gridspec.GridSpec(4, 2, figure=fig, height_ratios=[2.8, 1.8, 2.0, 1.6])
    ax_fc  = fig.add_subplot(gs[0, :])
    ax_att = fig.add_subplot(gs[1, 0])
    ax_sal = fig.add_subplot(gs[1, 1])
    ax_nli = fig.add_subplot(gs[2, :])
    ax_txt = fig.add_subplot(gs[3, :])

    # forecast
    h_tail  = history[-tail:]
    h_steps = np.arange(-len(h_tail), 0)
    f_steps = np.arange(HORIZON)
    p10 = np.asarray(result.forecast["p10"])
    p50 = np.asarray(result.forecast["p50"])
    p90 = np.asarray(result.forecast["p90"])
    ax_fc.plot(h_steps, h_tail, color="#555", lw=1.5)
    ax_fc.axvline(0, color="#aaa", lw=0.8, ls=":")
    ax_fc.fill_between(f_steps, p10, p90, alpha=0.25, color="#2980b9")
    ax_fc.plot(f_steps, p50, color="#2980b9", lw=2.2)
    ax_fc.plot(f_steps, future[:HORIZON], color="#e67e22", lw=1.5, ls="--")
    ax2 = ax_fc.twinx()
    for name in COV_NAMES:
        c = cov_set.values[-tail:, cov_set.names.index(name)]
        ax2.plot(h_steps, c, color=COV_COLORS[name], lw=0.9, alpha=0.55, ls="-.",
                 label=name.replace("_", " ").title())
    ax2.set_ylabel("Covariates", fontsize=7, color="#666")
    ax2.tick_params(labelsize=7)
    ax2.legend(loc="upper left", fontsize=7, framealpha=0.5)
    feat = result.features
    ax_fc.set_title(
        f"Trend: {feat.trend_magnitude} {feat.trend_direction}   "
        f"Uncertainty: {feat.uncertainty_level} ({feat.uncertainty_trend})   "
        f"Regime shift: {feat.regime_shift}   "
        f"Downside risk: {feat.downside_risk}   "
        f"Upside potential: {feat.upside_potential}", fontsize=8.5)
    ax_fc.set_xlabel("Steps relative to forecast start", fontsize=8)
    ax_fc.set_ylabel("Units sold", fontsize=8)
    ax_fc.legend(handles=[
        plt.Line2D([0],[0], color="#555",    lw=1.5,           label="History"),
        plt.Line2D([0],[0], color="#2980b9", lw=2.2,           label="P50 (median)"),
        mpatches.Patch(facecolor="#2980b9",  alpha=0.3,        label="P10\u2013P90"),
        plt.Line2D([0],[0], color="#e67e22", lw=1.5, ls="--", label="Actual"),
    ], loc="upper right", fontsize=8)
    ax_fc.tick_params(labelsize=8)

    # attribution
    attrs = result.attribution.attributions[:result.attribution.top_k]
    if attrs:
        names_a    = [a.name.replace("_", " ").title() for a in attrs]
        impacts    = [a.relative_impact_pct for a in attrs]
        bar_colors = [COV_COLORS.get(a.name, _CMAP(i)) for i, a in enumerate(attrs)]
        y = np.arange(len(names_a))
        bars = ax_att.barh(y, impacts, color=bar_colors, alpha=0.85)
        for bar, val in zip(bars, impacts):
            ax_att.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                        f"{val:.1f}%", va="center", fontsize=8)
        ax_att.set_yticks(y)
        ax_att.set_yticklabels(names_a, fontsize=8)
        ax_att.set_xlabel("Relative impact (%)", fontsize=8)
        ax_att.set_xlim(0, max(impacts) * 1.28)
    ax_att.set_title("Covariate Attribution — Attention Rollout", fontsize=9, fontweight="bold")
    ax_att.tick_params(labelsize=8)

    # temporal saliency
    temporal = result.attribution.temporal
    if temporal:
        steps = np.arange(len(temporal[0].saliency))
        for ta in temporal:
            c = COV_COLORS.get(ta.covariate_name, _CMAP(0))
            lbl = (f"{ta.covariate_name.replace('_',' ').title()} "
                   f"(peak={ta.peak_step}, breadth={ta.focus_breadth:.2f})")
            ax_sal.plot(steps, ta.saliency, color=c, lw=1.4, label=lbl)
            thr = np.percentile(ta.saliency, 75)
            ax_sal.fill_between(steps, 0, ta.saliency, where=ta.saliency >= thr,
                                color=c, alpha=0.20)
        ax_sal.set_xlabel("History step (0 = oldest)", fontsize=8)
        ax_sal.set_ylabel("Normalised attention", fontsize=8)
        ax_sal.legend(fontsize=6.5, loc="upper left", framealpha=0.6)
    else:
        ax_sal.text(0.5, 0.5, "No temporal saliency data",
                    ha="center", va="center", transform=ax_sal.transAxes, fontsize=9)
        ax_sal.axis("off")
    ax_sal.set_title("Temporal Saliency per Covariate", fontsize=9, fontweight="bold")
    ax_sal.tick_params(labelsize=8)

    # NLI
    report    = result.consistency_report
    sentences = report.sentence_scores
    ax_nli.axis("off")
    oc = _NLI_GREEN if report.is_consistent else _NLI_RED
    ax_nli.text(0.0, 1.0,
                f"NLI Consistency  overall={report.overall_score:.3f}  "
                f"[{'PASS' if report.is_consistent else 'FAIL'}]",
                transform=ax_nli.transAxes, fontsize=9, fontweight="bold",
                color=oc, va="top")
    n_sent = max(len(sentences), 1)
    row_h  = 0.85 / n_sent
    for i, ss in enumerate(sentences):
        y_pos = 0.88 - i * row_h
        sc    = ss.entailment_prob
        ax_nli.text(0.0,   y_pos, f"{sc:.2f}", transform=ax_nli.transAxes,
                    fontsize=7, color=_nli_color(sc), fontweight="bold", va="top")
        short = ss.sentence[:135] + "\u2026" if len(ss.sentence) > 135 else ss.sentence
        ax_nli.text(0.055, y_pos, short, transform=ax_nli.transAxes,
                    fontsize=7, color="#2c3e50", va="top")
    ax_nli.set_title("NLI Per-Sentence Consistency Scores", fontsize=9, fontweight="bold")

    # verbalization
    ax_txt.axis("off")
    ax_txt.text(0.0, 1.0, "Generated Summary:", transform=ax_txt.transAxes,
                fontsize=9, fontweight="bold", va="top", color="#2c3e50")
    wrapped = "\n".join(textwrap.wrap(result.verbalization.summary, width=165))
    ax_txt.text(0.0, 0.78, wrapped, transform=ax_txt.transAxes,
                fontsize=8, va="top", color="#2c3e50",
                bbox=dict(boxstyle="round,pad=0.5", fc="#ecf0f1", ec="#bdc3c7"))
    plt.show()

## 1 — Synthetic Sales Data

256 days of daily unit sales with three business covariates.

In [ ]:
N       = 256
HORIZON = 14
TOTAL   = N + HORIZON
t       = np.arange(TOTAL, dtype=float)

rng1 = np.random.RandomState(10)
s1_target = 200 + 1.55 * t + rng1.randn(TOTAL) * 8
s1_mkt    = 50  + 0.95 * t + rng1.randn(TOTAL) * 12
s1_promo  = 15  + 0.20 * t + rng1.randn(TOTAL) * 5
s1_comp   = 400             + rng1.randn(TOTAL) * 25

rng2 = np.random.RandomState(20)
s2_target = 600 - 1.80 * t + rng2.randn(TOTAL) * 8
s2_mkt    = 120 - 0.30 * t + rng2.randn(TOTAL) * 12
s2_promo  = 40  + 0.10 * t + rng2.randn(TOTAL) * 5
s2_comp   = 150 + 2.00 * t + rng2.randn(TOTAL) * 25

rng3 = np.random.RandomState(30)
s3_target = 300 + np.cumsum(rng3.randn(TOTAL) * 9)
s3_mkt    = 80  + rng3.randn(TOTAL) * 20
s3_promo  = 25  + rng3.randn(TOTAL) * 8
s3_comp   = 300 + rng3.randn(TOTAL) * 30

SCENARIOS = {
    "Growth":   make_split(s1_target, s1_mkt, s1_promo, s1_comp, N, HORIZON),
    "Decline":  make_split(s2_target, s2_mkt, s2_promo, s2_comp, N, HORIZON),
    "Volatile": make_split(s3_target, s3_mkt, s3_promo, s3_comp, N, HORIZON),
}

fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=False)
for ax, (name, data), color in zip(axes, SCENARIOS.items(),
                                    ["#27ae60", "#c0392b", "#8e44ad"]):
    h, fh, *_ = data
    ax.plot(np.arange(N),            h,  color=color, lw=1.5)
    ax.plot(np.arange(N, N+HORIZON), fh, color=color, lw=1.5, ls="--", alpha=0.5)
    ax.axvline(N, color="#aaa", lw=0.8, ls=":")
    ax.set_title(f"Scenario: {name} — daily unit sales", fontsize=10, fontweight="bold")
    ax.set_ylabel("Units sold", fontsize=8)
    ax.tick_params(labelsize=7)
plt.suptitle("Synthetic Sales Scenarios", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 2 — Pipeline Initialisation

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

provider   = ChronosForecastProvider(device=device, enable_attention=True)
verbalizer = TemplateVerbalizer(seed=42)
scorer     = NLIConsistencyScorer()

config = PipelineConfig(
    horizon=HORIZON,
    moderate_threshold=0.002,
    sharp_threshold=0.005,
    downside_factor=0.92,
    upside_factor=1.08,
)
pipeline = VerbalizationPipeline(
    forecast_provider=provider,
    verbalizer=verbalizer,
    scorer=scorer,
    config=config,
)
print("Pipeline ready.")

## 3 — Run All Scenarios

In [ ]:
demo_attributor = AttentionAttributor(top_k=5, temporal_entropy_threshold=1.01)

results = {}
for name, (history, future, past_cov, future_cov) in SCENARIOS.items():
    print(f"Running: {name} ...", end=" ", flush=True)
    cov_set        = CovariateSet.from_dict(past_cov)
    future_cov_set = CovariateSet.from_dict(future_cov)
    result = pipeline.run(
        history, horizon=HORIZON,
        covariates=cov_set, future_covariates=future_cov_set,
    )
    full_attr = demo_attributor.explain(cov_set, attention_weights=result.attention_weights)
    result.attribution.temporal = full_attr.temporal
    results[name] = (result, history, future, cov_set)
    f, rep = result.features, result.consistency_report
    print(f"done  |  {f.trend_magnitude} {f.trend_direction}  "
          f"unc={f.uncertainty_level}  "
          f"NLI={rep.overall_score:.3f} ({'PASS' if rep.is_consistent else 'FAIL'})")

## 4 — Scenario 1: Growth

Product in launch phase — sales rising ~1.55 units/day, `marketing_spend` strongly correlated.

In [ ]:
result, history, future, cov_set = results["Growth"]
plot_scenario(result, history, future, cov_set, "Growth", HORIZON)

## 5 — Scenario 2: Decline

Product losing market share — sales falling ~1.80 units/day while competitor grows.

In [ ]:
result, history, future, cov_set = results["Decline"]
plot_scenario(result, history, future, cov_set, "Decline", HORIZON)

## 6 — Scenario 3: Volatile Market

Sales follow a random walk — no trend, covariates independent of the target.

In [ ]:
result, history, future, cov_set = results["Volatile"]
plot_scenario(result, history, future, cov_set, "Volatile", HORIZON)